In [ ]:
import os, sys
os.chdir("/kaggle/working")
if not os.path.exists("CubiCasa5k"):
    !git clone --depth 1 https://github.com/CubiCasa/CubiCasa5k.git
os.chdir("/kaggle/working/CubiCasa5k")
sys.path.insert(0, os.getcwd())

# Gotcha 2: neutralize the removed scipy import (it's unused anyway)
!sed -i 's/^from scipy.ndimage import measurements/import scipy.ndimage as measurements/' floortrans/post_prosessing.py

# Pretrained weights from the repo's Google Drive
!pip install -q gdown
!gdown 1gRB7ez1e4H7a9Y09lLqRuna0luZO5VRK -O model_best_val_loss_var.pkl
print("setup done:", os.listdir(".")[:6])

In [ ]:
import torch
from floortrans.models import get_model

model = get_model('hg_furukawa_original', 51)
n_classes = 44
model.conv4_   = torch.nn.Conv2d(256, n_classes, bias=True, kernel_size=1)
model.upsample = torch.nn.ConvTranspose2d(n_classes, n_classes, kernel_size=4, stride=4)

ckpt = torch.load('model_best_val_loss_var.pkl', map_location='cuda', weights_only=False)  # Gotcha 1
model.load_state_dict(ckpt['model_state'])
model.eval().cuda()
print("model loaded")

In [ ]:
# Compat shim: make scipy.stats.mode behave like the version this repo expects
from scipy import stats
_orig_mode = stats.mode
def _mode_compat(a, axis=0, **kw):
    kw.setdefault("keepdims", True)   # restores array-shaped .mode so .mode[0] works
    return _orig_mode(a, axis=axis, **kw)
stats.mode = _mode_compat
print("scipy.stats.mode patched")

In [ ]:
import numpy as np, torch.nn.functional as F
from PIL import Image
from floortrans.post_prosessing import split_prediction, get_polygons

img_path = "/kaggle/input/datasets/qmarva/cubicasa5k/cubicasa5k/cubicasa5k/colorful/10052/F1_original.png"
arr = np.asarray(Image.open(img_path).convert("RGB"), dtype=np.float32) / 255.0
H, W = arr.shape[:2]

image = torch.tensor(2*arr - 1).permute(2,0,1).unsqueeze(0).cuda()   # normalize to [-1,1]
split = [21, 12, 11]
with torch.no_grad():
    pred = model(image)
    pred = F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=True)
    pred = pred.cpu()

heatmaps, rooms, icons = split_prediction(pred, (H, W), split)
polygons, types, room_polygons, room_types = get_polygons((heatmaps, rooms, icons), 0.1, [1, 2])
print(f"image {W}x{H} | polygons: {len(polygons)} | rooms: {len(room_polygons)}")

In [ ]:
import json, numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry.base import BaseMultipartGeometry

room_classes = ["Background","Outdoor","Wall","Kitchen","Living Room","Bed Room",
                "Bath","Entry","Railing","Storage","Garage","Undefined"]
DEFAULT_WALL_HEIGHT_M = 2.7

def rect_to_centerline(poly):
    pts = np.asarray(poly, dtype=float); n = len(pts)
    edges = [(i,(i+1)%n) for i in range(n)]
    L = [np.linalg.norm(pts[a]-pts[b]) for a,b in edges]
    o = np.argsort(L)                               # two shortest edges = the ends
    m1 = pts[list(edges[o[0]])].mean(0); m2 = pts[list(edges[o[1]])].mean(0)
    return m1.tolist(), m2.tolist(), float((L[o[0]]+L[o[1]])/2), float(max(L))

walls, openings = [], []
for i, t in enumerate(types):
    poly = polygons[i]
    if t['type'] == 'wall':
        s, e, thick, _ = rect_to_centerline(poly)
        walls.append({"id": f"w{i}", "start": s, "end": e,
                      "thickness_px": round(thick,1), "height_m": DEFAULT_WALL_HEIGHT_M})
    elif t['type'] == 'icon' and int(t['class']) in (1, 2):   # 1=window, 2=door
        c = np.asarray(poly, dtype=float).mean(0)
        _, _, _, width = rect_to_centerline(poly)
        openings.append({"id": f"o{i}", "type": "window" if int(t['class'])==1 else "door",
                         "wall_id": None, "position": [round(c[0],1), round(c[1],1)],
                         "width_px": round(width,1)})

def polygon_rings(geom, min_area=1.0):
    """Recursively collect exterior rings from any geometry; skip non-polygons and slivers."""
    rings = []
    if geom.is_empty:
        return rings
    if isinstance(geom, Polygon):
        if geom.area >= min_area:
            rings.append([[round(float(x),1), round(float(y),1)] for x, y in geom.exterior.coords])
    elif isinstance(geom, BaseMultipartGeometry):   # MultiPolygon, GeometryCollection, etc.
        for part in geom.geoms:
            rings.extend(polygon_rings(part, min_area))
    # anything else (LineString, Point) is silently skipped
    return rings

rooms = []
for j, (rp, rt) in enumerate(zip(room_polygons, room_types)):
    cls = int(rt['class'])
    if cls in (0, 1, 2):   # skip Background / Outdoor / Wall
        continue
    for k, ring in enumerate(polygon_rings(rp)):
        rooms.append({"id": f"r{j}_{k}", "label": room_classes[cls], "polygon": ring})

geometry = {
    "schema_version": "1.0",
    "source": {"image_width_px": W, "image_height_px": H},
    "scale": {"pixels_per_meter": 100.0, "source": "assumed_default", "confidence": 0.0},
    "origin_px": [0.0, 0.0],
    "walls": walls, "openings": openings, "rooms": rooms,
}
with open("/kaggle/working/sample_geometry.json", "w") as f:
    json.dump(geometry, f, indent=2)
print(f"walls={len(walls)} openings={len(openings)} rooms={len(rooms)} -> sample_geometry.json")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 8))
for w in geometry["walls"]:
    (x1, y1), (x2, y2) = w["start"], w["end"]
    ax.plot([x1, x2], [y1, y2], linewidth=2)
ax.set_aspect("equal"); ax.invert_yaxis()
ax.set_title(f'{len(geometry["walls"])} walls')
plt.show()

In [ ]:
import json, numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry.base import BaseMultipartGeometry

RATIO = 0.01   # PLACEHOLDER pixel->meter (~100 px/m). Provisional until scale is calibrated by the AR team.

room_classes = ["Background","Outdoor","Wall","Kitchen","Living Room","Bed Room",
                "Bath","Entry","Railing","Storage","Garage","Undefined"]

def norm_label(lbl):
    m = {"Bed Room":"bedroom","Living Room":"living_room","Bath":"bathroom","Kitchen":"kitchen",
         "Entry":"entry","Storage":"storage","Garage":"garage","Undefined":"room"}
    return m.get(lbl, lbl.lower().replace(" ","_"))

def polygon_rings(geom, min_area=1.0):
    rings=[]
    if geom.is_empty: return rings
    if isinstance(geom, Polygon):
        if geom.area >= min_area:
            rings.append([{"x":round(float(x),1),"y":round(float(y),1)} for x,y in geom.exterior.coords])
    elif isinstance(geom, BaseMultipartGeometry):
        for part in geom.geoms: rings.extend(polygon_rings(part, min_area))
    return rings

# --- rooms[] ---
rooms=[]
for j,(rp,rt) in enumerate(zip(room_polygons, room_types)):
    cls=int(rt['class'])
    if cls in (0,1,2): continue                      # skip Background/Outdoor/Wall
    for k,ring in enumerate(polygon_rings(rp)):
        rooms.append({"id":f"room_{j:03d}_{k}", "type":norm_label(room_classes[cls]), "polygon":ring})

# --- fixtures[] (doors/windows from the icon head) ---
fixtures=[]; d=w=0
for i,t in enumerate(types):
    if t['type']=='icon' and int(t['class']) in (1,2):
        pts=np.asarray(polygons[i],float); cx,cy=pts.mean(0)
        x_ext=pts[:,0].max()-pts[:,0].min(); y_ext=pts[:,1].max()-pts[:,1].min()
        width_px=max(x_ext,y_ext); rotation=90.0 if y_ext>x_ext else 0.0
        is_door=int(t['class'])==2
        if is_door: d+=1; fid=f"door_{d:03d}"; ftype="door"; hgt=2.1
        else:       w+=1; fid=f"window_{w:03d}"; ftype="window"; hgt=1.2
        fixtures.append({"id":fid,"type":ftype,
                         "position":{"x":round(float(cx),1),"y":round(float(cy),1)},
                         "scale":{"width":round(float(width_px*RATIO),2),"depth":0.1,"height":hgt},
                         "rotation_degrees":rotation})

fragment={"metadata":{"original_width_px":int(W),"original_height_px":int(H),"pixel_to_meter_ratio":RATIO},
          "walls":[], "rooms":rooms, "fixtures":fixtures}
with open("/kaggle/working/cubicasa_rooms_fixtures.json","w") as f:
    json.dump(fragment,f,indent=2)
print(f"rooms={len(rooms)} fixtures={len(fixtures)} (doors={d}, windows={w}) -> cubicasa_rooms_fixtures.json")